# speed from smooth data 

In [29]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

## compute speed csv from smooth data

In [30]:
# add extra linear interpolation for head and wing bases

def scipy_interpolate_1d(x, max_gap=40):
    from scipy.interpolate import interp1d 
    """
    Linear interpolation using scipy.interpolate.interp1d
    Interpolates NaN gaps <= max_gap
    Does NOT interpolate at start or end
    
    @2500 fps, and 250Hz wbf - a gap of 40 is about 4 wingbeats.
    """
    y = x.copy()
    n = len(x)

    isnan = np.isnan(x)
    padded = np.r_[False, isnan, False]
    diff = np.diff(padded.astype(int))

    starts = np.where(diff == 1)[0]
    ends   = np.where(diff == -1)[0]

    for s, e in zip(starts, ends):
        gap_len = e - s

        if gap_len > max_gap:
            continue

        if s == 0 or e == n:
            continue

        idx = np.array([s - 1, e])
        vals = y[idx]

        if np.any(np.isnan(vals)):
            continue

        f = interp1d(idx, vals, kind='linear')
        y[s:e] = f(np.arange(s, e))

    return y

In [31]:
def savgol_smoothening(x, window = 31, poly = 3):
    """
    Savitzky–Golay smoothening on continuous (non-NaN) segments only.
    Long NaN blocks are preserved.

    x      : 1D array
    dt     : time step (seconds)
    window : odd window length
    poly   : polynomial order
    """
    pos = np.full_like(x, np.nan)

    isnan = np.isnan(x)
    idx = np.arange(len(x))

    # split into continuous non-NaN segments
    segments = np.split(idx, np.where(isnan)[0])

    for seg in segments:
        if len(seg) < window +2 :
            continue
            
        xs = x[seg]
        
        if np.any(~np.isfinite(xs)):
            continue

        if np.nanstd(xs) < 1e-8:
            continue

        pos[seg] = savgol_filter(
            xs,
            window_length=window,
            polyorder=poly,
            mode='interp'
        )

    return pos

In [32]:
def largest_nan_segment(mask):
    """
    mask: 1D boolean array, True = NaN frame
    """

    # Pad to catch edges
    padded = np.r_[False, mask, False]
    diff = np.diff(padded.astype(int))

    starts = np.where(diff == 1)[0]
    ends   = np.where(diff == -1)[0]

    lengths = ends - starts

    return lengths

In [33]:
def plot_interpolated(raw, interpolated):
    import matplotlib.pyplot as plt
    
    f, ax = plt.subplots(2,1, sharex = True)
    ax= ax.ravel()
    
    ax[0].plot(raw)
    ax[1].plot(interpolated)
    return f

In [34]:
# Input 
INPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/xyz_Smooth"
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Speed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Optional frame limits 
begin_frame = None   # e.g. 0
end_frame   = None   # e.g. 650

interpol_check = False

### Savitsky parameters for smoothening ###
smooth_window = 41           # odd, tune based on sampling
poly = 3
fps = 2500
dt  = 1 / fps
##########################################################

# Find smooth files
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv"))
if not csv_files:
    raise FileNotFoundError("No SMOOTH CSV files found.")

In [39]:
for path in csv_files:
    fname = os.path.basename(path)
    print("Processing:", fname)

    df = pd.read_csv(path)

    if 'frame' not in df.columns:
        df['frame'] = np.arange(len(df))

    frames = df["frame"].values
    center = np.column_stack([
        df["center_X"].values,
        df["center_Y"].values,
        df["center_Z"].values
    ])
    
    
    # ------------------------------------
    # extra Linear interpolation + smoothening the center of the fly
    # ------------------------------------
    
    center_interp = center.copy()

    for i in range(3):
        center_interp[:, i] = scipy_interpolate_1d(
            center[:, i],
            max_gap=20
            )
    
    if interpol_check == True:
        f = plot_interpolated(center, center_interp)
        f.savefig('./../../Tanvi-files/Figures/Check_Smoothening/' + 
                fname + 'interpol.png')
    D = center.shape[1]   # should be 3
    
    
    center_smooth = center_interp.copy()
    for i in range(3):
        center_smooth[:,i] = savgol_smoothening(
        center_interp[:,i]
            , window = smooth_window
            , poly = 3
        )
    # ----------------------------------
    # compute velocity and acceleration
    # ----------------------------------
    
    velocity = np.gradient(center_smooth, dt, axis = 0)
    speed = np.linalg.norm(velocity, axis=1)
    
    # ---------------------------------------
    # SAVE DATA 
    # ---------------------------------------
    
    out_df = pd.DataFrame({
        'frame': frames,
        'posx': center_smooth[:,0],
        'posy': center_smooth[:,1],
        'posz': center_smooth[:,2],
        'vx': velocity[:, 0],
        'vy': velocity[:, 1],
        'vz': velocity[:, 2],
        'speed': speed,
    })


    out_name = fname.replace("_SMOOTH.csv", "_SPEED.csv")
    out_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)

    print("Saved:", out_name)

print("\n Speed CSV generation complete.")

Processing: Trial2_180rpmxyzpts_SMOOTH.csv
Saved: Trial2_180rpmxyzpts_SPEED.csv
Processing: Trial2_200rpmxyzpts_SMOOTH.csv
Saved: Trial2_200rpmxyzpts_SPEED.csv
Processing: Trial3_180rpmxyzpts_SMOOTH.csv
Saved: Trial3_180rpmxyzpts_SPEED.csv
Processing: Trial4_400rpmxyzpts_SMOOTH.csv
Saved: Trial4_400rpmxyzpts_SPEED.csv
Processing: Trial5_180rpmxyzpts_SMOOTH.csv
Saved: Trial5_180rpmxyzpts_SPEED.csv
Processing: Trial5_400rpmxyzpts_SMOOTH.csv
Saved: Trial5_400rpmxyzpts_SPEED.csv
Processing: Trial7_400rpmxyzpts_SMOOTH.csv
Saved: Trial7_400rpmxyzpts_SPEED.csv

 Speed CSV generation complete.
